In [ ]:
# Cell 1 — Setup + data load: US500 30m panel from Dukascopy 1-min BID closes.
# 1-min bars are open-time-stamped; resampling with label="right" stamps each 30m bar
# by its CLOSE time -> PIT-honest at build (the #011 lesson, applied up front).
import pandas as pd
from backtest.data import DataPanel

RAW = pd.read_parquet(r"C:\Users\User\backtest_engine\backtest_engine2\data\us500_1m.parquet")
s = RAW.set_index("open_time")["close"].sort_index()
s.index = s.index.tz_localize(None)  # tz-naive UTC

px30 = s.resample("30min", closed="left", label="right").last().dropna()
prices = px30.to_frame("USA500")

panel = DataPanel(prices, check_outliers=True)

print(f"Bars: {len(panel.dates)}")
print(f"Range: {panel.dates[0]} -> {panel.dates[-1]}")
print("Bars per year:", prices.groupby(prices.index.year).size().to_dict())
bars_per_day = prices.groupby(prices.index.normalize()).size()
print(f"Days with data: {len(bars_per_day)}; bars/day median: {bars_per_day.median():.0f}")
print(f"NaN fraction: {prices['USA500'].isna().mean():.3%}")
prices.tail(3)


In [ ]:
# Cell 2 — Stage 1 amendment: rebuild panel from the gap-repaired parquet.
# (Puller's vol>0 filter had dropped 2022-07-26..2023-05-19 — volume field was zeroed
# server-side in that window. Re-pulled with price-change-based activity filter, merged.)
# Outliers (COVID March 2020) reviewed and accepted in Cell 1 -> check_outliers=False.
RAW = pd.read_parquet(r"C:\Users\User\backtest_engine\backtest_engine2\data\us500_1m.parquet")
s = RAW.set_index("open_time")["close"].sort_index()
s.index = s.index.tz_localize(None)

px30 = s.resample("30min", closed="left", label="right").last().dropna()
prices = px30.to_frame("USA500")

panel = DataPanel(prices, check_outliers=False)

print(f"Bars: {len(panel.dates)}")
print(f"Range: {panel.dates[0]} -> {panel.dates[-1]}")
print("Bars per year:", prices.groupby(prices.index.year).size().to_dict())
bars_per_day = prices.groupby(prices.index.normalize()).size()
print(f"Days with data: {len(bars_per_day)}; bars/day median: {bars_per_day.median():.0f}")
prices.tail(3)


In [ ]:
# Cell 3 — Stage 2: paper-window overnight drift (trial 1, pre-registered replication).
# Long US500 2:00-3:00am ET every night (Boyarchenko/Larsen/Whelan SR917 headline window).
# Signal-free unconditional premium harvest; ET-anchored so DST is handled by tz conversion.
import numpy as np
from backtest.strategy import Strategy
from backtest.risk import RiskConfig

ET = "America/New_York"


def _et_hour(dates):
    return dates.tz_localize("UTC").tz_convert(ET)


def _od_rebalance_bars(dates):
    et = _et_hour(dates)
    pick = ((et.hour == 2) | (et.hour == 3)) & (et.minute == 0)
    return dates[pick]


class OvernightDriftWindow(Strategy):
    """SR917 replication: long the 2-3am ET window, flat otherwise."""
    rebalance_frequency = staticmethod(_od_rebalance_bars)

    def __init__(self, gross: float = 1.0):
        self.gross = gross
        self.risk = RiskConfig(max_position=gross, max_gross=1.0)

    def __repr__(self):
        return f"OvernightDriftWindow(window=02:00-03:00ET, long_only, gross={self.gross})"

    def generate_weights(self, data, t):
        et_t = t.tz_localize("UTC").tz_convert(ET)
        if et_t.hour == 2 and et_t.minute == 0:
            return pd.Series({"USA500": self.gross})
        return pd.Series(0.0, index=data.assets)


strat = OvernightDriftWindow(gross=1.0)
print(strat)
rb = strat.rebalance_dates(panel.dates)
years = (panel.dates[-1] - panel.dates[0]).days / 365.25
et_rb = _et_hour(rb)
print(f"Rebalance bars: {len(rb)} total, {len(rb) / years:.0f}/yr (expect ~2x250 = ~500/yr)")
print(f"  entries (ET 2:00): {int((et_rb.hour == 2).sum())}")
print(f"  exits   (ET 3:00): {int((et_rb.hour == 3).sum())}")


In [ ]:
# Cell 4 — Stage 3: spread-only cost stack (FTMO index CFD).
# No commission (FTMO indices are commission-free). No MarketImpact/LiquidityCap (no real
# volume data; trade size trivial vs ES depth — decided Cell 1). half_bps=1.0 ~= 0.5pt at
# S&P 5000; 0.5/2.0bp sensitivity at Stage 7 covers the points-vs-bps drift.
from backtest.costs import Spread, CompositeCostModel

costs = CompositeCostModel([Spread(half_bps=1.0)])
liq = None

print("Cost components:", [type(m).__name__ for m in costs.models])
print("Round-trip cost: ~2.0 bp/night vs paper drift ~1.4 bp/night (the gate under test)")


In [ ]:
# Cell 5 — Stage 4: single-path run, trial 1 (paper window 2-3am ET, long-only).
from backtest.engine import Engine

WARMUP_BARS = 48
engine = Engine(panel, strat, costs=costs, liquidity=None, initial_capital=1_000_000)
result = engine.run(start=panel.dates[WARMUP_BARS])

print(f"Bars:          {len(result.equity_curve)}")
print(f"Rebalances:    {result.metadata['n_rebalances']}")
print(f"Final equity:  ${result.equity_curve.iloc[-1]:>14,.0f}")
print(f"Total return:  {(result.equity_curve.iloc[-1] / 1_000_000 - 1):+.2%}")
print(f"Total costs:   ${result.costs.sum():>14,.0f}")
print(f"Max position weight:  {result.weights.abs().max().max():.2%}")

# Gross per-night quality: window return = P(exit bar)/P(entry bar) - 1 on held nights.
px_ = prices["USA500"]
held = result.positions["USA500"]
entry_stamps = held.index[(held.abs() > 1.0).values & (_et_hour(held.index).hour == 2)]
exit_stamps = entry_stamps + pd.Timedelta("1h")
ret_n = pd.Series(px_.reindex(exit_stamps).values / px_.reindex(entry_stamps).values - 1.0,
                  index=entry_stamps).dropna()
sh_n = ret_n.mean() / ret_n.std() * np.sqrt(250)
print(f"\nGross window: {len(ret_n)} nights | hit {100 * (ret_n > 0).mean():.1f}% | "
      f"mean {1e4 * ret_n.mean():+.2f} bps/night | ann {250 * ret_n.mean():+.2%}/yr | "
      f"gross Sharpe {sh_n:+.2f}")
print("Paper benchmark: ~+1.4 bps/night, ~+3.6%/yr (1998-2019) | cost gate ~2.0 bp/night")


In [ ]:
# Cell 6 — Stage 5 diagnostic: corrected gross window measurement + per-year decay.
# Cell 5's filter caught hour==2 without minute==0 -> double-counted 2:00 and 2:30 stamps
# (3,829 "nights" ~= 2x1,916), blending the true window with an overlapping 2:30-3:30 one.
# Fix: entry = held bar stamped exactly 02:00 ET; exit = entry + 1h. ~1,916 true nights.
px_ = prices["USA500"]
held = result.positions["USA500"]
et_idx = _et_hour(held.index)
entry_stamps = held.index[(held.abs() > 1.0).values
                          & (et_idx.hour == 2) & (et_idx.minute == 0)]
exit_stamps = entry_stamps + pd.Timedelta("1h")
ret_n = pd.Series(px_.reindex(exit_stamps).values / px_.reindex(entry_stamps).values - 1.0,
                  index=entry_stamps).dropna()

sh = ret_n.mean() / ret_n.std() * np.sqrt(250)
print(f"True window (2:00->3:00 ET): {len(ret_n)} nights | hit {100 * (ret_n > 0).mean():.1f}% | "
      f"mean {1e4 * ret_n.mean():+.2f} bps/night | ann {250 * ret_n.mean():+.2%}/yr | "
      f"gross Sharpe {sh:+.2f}")
print("Paper benchmark: ~+1.4 bps/night, ~+3.6%/yr (1998-2019) | cost gate ~2.0 bp/night\n")

# Per-year decay table (the #011-style read: decayed vs never-worked).
print(f"{'year':>5} {'nights':>7} {'hit%':>6} {'bps/nt':>8} {'gSharpe':>8}")
for yr, grp in ret_n.groupby(ret_n.index.year):
    shy = grp.mean() / grp.std() * np.sqrt(250) if grp.std() > 0 else float("nan")
    print(f"{yr:>5} {len(grp):>7} {100 * (grp > 0).mean():>6.1f} "
          f"{1e4 * grp.mean():>+8.2f} {shy:>+8.2f}")


In [ ]:
# Cell 7 — Stage 5 diagnostic: hour-of-night decomposition.
# Where does overnight drift live now? Paper (SR917) said the premium migrated across
# hours over its sample. For each 1h window h:00->h+1:00 ET through the non-RTH session,
# mean bps, hit%, gross Sharpe — split early (2019-2021) vs late (2022-2026).
px_ = prices["USA500"]
et_all = _et_hour(px_.index)
HOURS = [18, 19, 20, 21, 22, 23, 0, 1, 2, 3, 4, 5, 6, 7, 8, 9]  # futures reopen -> RTH open

def _hour_ret(h):
    ent = px_.index[(et_all.hour == h) & (et_all.minute == 0)]
    ext = ent + pd.Timedelta("1h")
    return pd.Series(px_.reindex(ext).values / px_.reindex(ent).values - 1.0,
                     index=ent).dropna()

def _stats(r):
    if len(r) < 50:
        return None
    return (len(r), 100 * (r > 0).mean(), 1e4 * r.mean(),
            r.mean() / r.std() * np.sqrt(250) if r.std() > 0 else float("nan"))

print(f"{'ET hr':>6} | {'n':>5} {'hit%':>5} {'bps':>6} {'gSh':>6}  <- 2019-2021 | "
      f"{'n':>5} {'hit%':>5} {'bps':>6} {'gSh':>6}  <- 2022-2026")
for h in HOURS:
    r = _hour_ret(h)
    early = _stats(r[r.index.year <= 2021])
    late = _stats(r[r.index.year >= 2022])
    fe = f"{early[0]:>5} {early[1]:>5.1f} {early[2]:>+6.2f} {early[3]:>+6.2f}" if early else " " * 25
    fl = f"{late[0]:>5} {late[1]:>5.1f} {late[2]:>+6.2f} {late[3]:>+6.2f}" if late else " " * 25
    print(f"{h:>4}:00 | {fe} | {fl}")
print("\ncost gate ~2.0 bp/window (1bp half-spread x2) | paper window = 2:00")


In [ ]:
# Cell 8 — #013 trial 1 verdict: KILLED (decayed).
# Pre-registered window (SR917, long 2:00-3:00am ET): gross +0.47 bp/night vs paper +1.4,
# vs cost gate ~2.0. Per-year: only 2020 (+4.13 bp) positive at scale; dead 2021-2025.
# Hour map (Cell 7): no late-period hour clears costs; premium is a diffuse smear.
# Spread sensitivity skipped as moot: gross fails even a 0.5 bp half-spread world.
print("#013 trial 1 (2:00-3:00am ET window): KILLED - decayed post-publication")
print("  gross +0.47 bp/night (paper +1.4) | cost gate ~2.0 bp | only 2020 positive")
print("  trial 2 (pre-registered): full close->open hold, swap-parameterized")
print("  rationale: late-period hours sum to ~+2.4 bp/night for ONE round trip (~2.0 bp)")


In [ ]:
# Cell 9 — #013 trial 2 (pre-registered): full close->open hold, swap-parameterized.
# Long at the 16:00 ET bar (cash close), flat at the 9:30 ET bar (cash open). One round
# trip per night collects the diffuse hour-smear from Cell 7. Crosses the ~17:00 ET swap
# cutoff every night -> swap applies; swept as a parameter since real FTMO points unknown.
# Friday entries hold to Monday 9:30 (weekend gap + ~3 swap days) — kept, as in the paper.

def _cv_rebalance_bars(dates):
    et = _et_hour(dates)
    pick = ((et.hour == 16) & (et.minute == 0)) | ((et.hour == 9) & (et.minute == 30))
    return dates[pick]


class CloseToOpenHold(Strategy):
    """Trial 2: hold US500 from cash close (16:00 ET) to cash open (9:30 ET)."""
    rebalance_frequency = staticmethod(_cv_rebalance_bars)

    def __init__(self, gross: float = 1.0):
        self.gross = gross
        self.risk = RiskConfig(max_position=gross, max_gross=1.0)

    def __repr__(self):
        return f"CloseToOpenHold(16:00->9:30 ET, long_only, gross={self.gross})"

    def generate_weights(self, data, t):
        et_t = t.tz_localize("UTC").tz_convert(ET)
        if et_t.hour == 16 and et_t.minute == 0:
            return pd.Series({"USA500": self.gross})
        return pd.Series(0.0, index=data.assets)


strat2 = CloseToOpenHold(gross=1.0)
print(strat2)
rb2 = strat2.rebalance_dates(panel.dates)
et_rb2 = _et_hour(rb2)
print(f"Rebalance bars: {len(rb2)} | entries 16:00: {int((et_rb2.hour == 16).sum())} | "
      f"exits 9:30: {int((et_rb2.hour == 9).sum())}")

engine2 = Engine(panel, strat2, costs=costs, liquidity=None, initial_capital=1_000_000)
result2 = engine2.run(start=panel.dates[48])
print(f"\nEngine (spread-only, no swap): final ${result2.equity_curve.iloc[-1]:,.0f} | "
      f"return {result2.equity_curve.iloc[-1] / 1e6 - 1:+.2%} | "
      f"costs ${result2.costs.sum():,.0f} | rebalances {result2.metadata['n_rebalances']}")

# Gross close->open window return per held night (entry 16:00 -> next 9:30 stamp).
held2 = result2.positions["USA500"]
et_h2 = _et_hour(held2.index)
ent2 = held2.index[(held2.abs() > 1.0).values & (et_h2.hour == 16) & (et_h2.minute == 0)]
opens = px_.index[(et_all.hour == 9) & (et_all.minute == 30)]
nxt = opens.searchsorted(ent2)
valid = nxt < len(opens)
ent2, ext2 = ent2[valid], opens[nxt[valid]]
gross_n = pd.Series(px_.reindex(ext2).values / px_.reindex(ent2).values - 1.0,
                    index=ent2).dropna()
sh2 = gross_n.mean() / gross_n.std() * np.sqrt(250)
print(f"\nGross close->open: {len(gross_n)} nights | hit {100 * (gross_n > 0).mean():.1f}% | "
      f"mean {1e4 * gross_n.mean():+.2f} bps/night | ann {250 * gross_n.mean():+.2%}/yr | "
      f"gross Sharpe {sh2:+.2f}")

print(f"\n{'year':>5} {'nights':>7} {'hit%':>6} {'bps/nt':>8} {'gSharpe':>8}")
for yr, grp in gross_n.groupby(gross_n.index.year):
    shy = grp.mean() / grp.std() * np.sqrt(250) if grp.std() > 0 else float("nan")
    print(f"{yr:>5} {len(grp):>7} {100 * (grp > 0).mean():>6.1f} "
          f"{1e4 * grp.mean():>+8.2f} {shy:>+8.2f}")

# Swap sweep: net/night = gross - 2.0 bp round-trip spread - swap. Fri->Mon holds incur
# ~3 swap days; sweep values are per-HELD-night, so read FTMO's points as wkly avg /5.
print(f"\nSwap sweep (net = gross - 2.0 bp spread - swap), full period and 2022+:")
print(f"{'swap bp/nt':>11} {'net bps/nt':>11} {'net %/yr':>9} {'net Sh':>7} | {'2022+ bps':>10} {'2022+ Sh':>9}")
late = gross_n[gross_n.index.year >= 2022]
for swap in [0.0, 0.5, 1.0, 2.0]:
    net = gross_n - 2.0e-4 - swap * 1e-4
    netl = late - 2.0e-4 - swap * 1e-4
    print(f"{swap:>11.1f} {1e4 * net.mean():>+11.2f} {250 * net.mean():>+9.2%} "
          f"{net.mean() / net.std() * np.sqrt(250):>+7.2f} | "
          f"{1e4 * netl.mean():>+10.2f} {netl.mean() / netl.std() * np.sqrt(250):>+9.2f}")


In [ ]:
# Cell 10 — Stage 9 DSR + #013 session verdict.
# Best trial this session = trial 2 (close->open, spread-only engine run, NO swap —
# the most favorable case). K=2 pre-registered trials. DSR on engine bar returns.
from backtest.selection import dsr, expected_max_sr
from backtest.metrics.core import sharpe_var_term

ret2 = result2.returns
ANN = 252 * 45  # ~45 30m bars/day median
r = ret2.dropna().to_numpy()
sr_pb = r.mean() / r.std(ddof=1)
var_term = sharpe_var_term(r, sr_pb)
print(f"Trial 2 engine returns: ann Sharpe {sr_pb * np.sqrt(ANN):+.2f} (spread-only, swap=0)")
for K in [1, 2, 6]:
    print(f"  K={K}: luck-max {expected_max_sr(K, var_term / len(r)) * np.sqrt(ANN):+.3f} ann "
          f"| DSR {dsr(ret2, K=K):.3f}")
print("(K=2 = this session's pre-registered trials; K=6 = #011+#013 combined)")

print("\n#013 VERDICT: KILLED (both trials)")
print("  trial 1 (2-3am window): decayed — only 2020 positive, dead 2021+; gross < costs")
print("  trial 2 (close->open):  gross +2.75 bp/nt real, but 2022+ nets ~0 at swap=0;")
print("                          any positive swap or spread > 1bp half makes it negative")
print("  swap lookup NO LONGER GATES #013 — dead even in the best swap world")
